# Fine-Tuning MedSAM2 on APIS Brain CT Infarct Dataset

This notebook runs GPU fine-tuning for **MedSAM2** on the APIS dataset.

### Easy 2-Step Setup:
1. **Upload Archive:** Upload `apis_finetune_colab_pack.zip` (113 MB, located in your local `Code files` folder) into the Colab Files tab on the left.
2. **Run All Cells:** Run the cells sequentially. In ~30 minutes, it will train MedSAM2 on a T4 GPU and download `best_medsam2_apis.pt`.

In [ ]:
# Step 1: Verify GPU Acceleration
!nvidia-smi

In [ ]:
# Step 2: Install Required Dependencies
!pip install -q hydra-core fvcore iopath nibabel opencv-python matplotlib

In [ ]:
# Step 3: Unpack the Training Package
# Make sure 'apis_finetune_colab_pack.zip' has been uploaded to the Colab files panel (left sidebar)
import os

zip_name = 'apis_finetune_colab_pack.zip'
if os.path.isfile(zip_name):
    print(f"Unpacking {zip_name}...")
    !unzip -q -o "{zip_name}"
    print("Unpacking complete!")
else:
    print("Archive not found in root. If you want to upload it directly now, run the cell below.")

In [ ]:
# (Optional Upload Helper) Run this cell if you haven't uploaded the zip file yet
from google.colab import files
import os

if not os.path.isfile('apis_finetune_colab_pack.zip'):
    print("Please select 'apis_finetune_colab_pack.zip' from your computer:")
    uploaded = files.upload()
    !unzip -q -o apis_finetune_colab_pack.zip
    print("Unpacked successfully!")
else:
    print("Dataset package is already ready.")

In [ ]:
# Step 4: Download Pretrained MedSAM2 Initialization Weights
import os
os.makedirs('MEDSAM for infarct/checkpoints', exist_ok=True)
ckpt_path = 'MEDSAM for infarct/checkpoints/MedSAM2_CTLesion.pt'

if not os.path.isfile(ckpt_path):
    print("Downloading MedSAM2_CTLesion.pt (148 MB) from Hugging Face...")
    !wget -q -O "{ckpt_path}" https://huggingface.co/wanglab/MedSAM2/resolve/main/MedSAM2_CTLesion.pt
    print("Download complete!")
else:
    print(f"Checkpoint verified at: {ckpt_path}")

# Ensure LayerNorm2d import is clean
!sed -i "s/from efficient_track_anything.modeling.efficienttam_utils import LayerNorm2d/from sam2.modeling.sam2_utils import LayerNorm2d/g" "/content/MEDSAM for infarct/sam2/modeling/backbones/image_encoder.py" 2>/dev/null || true

In [ ]:
# Step 5: Run GPU Fine-Tuning (40 Epochs)
!python "MEDSAM for infarct/training/train_apis.py" \
    --epochs 40 \
    --batch_size 4 \
    --lr 5e-5 \
    --device cuda

In [ ]:
# Step 6: Plot Training Loss & Validation Metrics History
import json
import os
import matplotlib.pyplot as plt

history_file = 'MEDSAM for infarct/checkpoints/finetuned/training_history.json'
if os.path.isfile(history_file):
    with open(history_file, 'r') as f:
        hist = json.load(f)

    epochs = [h['epoch'] for h in hist]
    losses = [h['train_loss'] for h in hist]
    dices  = [h['val_dice'] for h in hist]
    ious   = [h['val_iou'] for h in hist]

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, losses, 'b-o', label='Train Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, dices, 'g-o', label='Val Dice (DSC)')
    plt.plot(epochs, ious, 'm-s', label='Val IoU')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.title('Validation Performance')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Training history file not found yet.")

In [ ]:
# Step 7: Download the Fine-Tuned Checkpoint to Your Local Machine
from google.colab import files
import os

best_ckpt = 'MEDSAM for infarct/checkpoints/finetuned/best_medsam2_apis.pt'
if os.path.isfile(best_ckpt):
    print(f"Downloading {best_ckpt} to your local computer...")
    files.download(best_ckpt)
else:
    print("best_medsam2_apis.pt not found. Check training step above.")